# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

According to the UCI description and the Materials and Methods section of the accompanying paper (Moro et al., 2014), the data represent **17 marketing campaigns** conducted by a Portuguese banking institution. The campaigns span from **May 2008 to November 2010**, with the full dataset (bank-additional-full.csv) containing 41,188 phone contacts ordered by date.

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('data/bank-additional/bank-additional-full.csv', sep = ';')

In [ ]:
df.head()

### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



In [ ]:
# Check for missing values (stored as 'unknown' in categoricals per the description)
print("Missing values (NaN):")
print(df.isnull().sum())
print("\n'unknown' counts in categorical columns:")
cat_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'poutcome']
for c in cat_cols:
    if c in df.columns:
        print(f"  {c}: {(df[c] == 'unknown').sum()}")
print("\nData types:")
print(df.dtypes)
# Conclusion: No NaN; categorical 'unknown' can be kept as a category. 
# Numeric columns are read correctly; we may one-hot encode categoricals for modeling.

### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

In [ ]:
df.info()

**Business Objective:** Predict whether a client will subscribe to a **term deposit** (variable `y`: yes/no) based on bank client and campaign-related features. The goal is to support the marketing team by identifying which clients are most likely to subscribe, so that outreach (e.g., phone campaigns) can be prioritized and resources used more efficiently.

### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

In [ ]:
# Bank client features only (per problem description)
bank_client_cols = ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']
X_raw = df[bank_client_cols].copy()
y = (df['y'] == 'yes').astype(int)  # binary: 1 = subscribed, 0 = not

In [ ]:
# One-hot encode categorical features
X_encoded = pd.get_dummies(X_raw, columns=['job', 'marital', 'education', 'default', 'housing', 'loan'], drop_first=False)
X_encoded.head()

In [ ]:
X = X_encoded.values  # use encoded features; scaling will be done in pipelines to avoid data leakage
feature_names = X_encoded.columns.tolist()
print("Feature matrix shape:", X.shape)
print("Target distribution:\n", y.value_counts())

### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])

### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

In [ ]:
# Baseline: predict the majority class (no subscription)
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
print("Baseline (majority class) accuracy:", round(baseline_accuracy, 4))
print("Our classifiers should beat this to be useful.")

### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)  # simple model on bank-client features only

### Problem 9: Score the Model

What is the accuracy of your model?

In [ ]:
print("Logistic Regression - Train accuracy:", round(lr.score(X_train, y_train), 4))
print("Logistic Regression - Test accuracy:", round(lr.score(X_test, y_test), 4))

### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

In [ ]:
import time
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

def fit_and_score(name, model, X_tr, y_tr, X_te, y_te):
    start = time.perf_counter()
    model.fit(X_tr, y_tr)
    train_time = time.perf_counter() - start
    train_acc = model.score(X_tr, y_tr)
    test_acc = model.score(X_te, y_te)
    return {'Model': name, 'Train Time': train_time, 'Train Accuracy': train_acc, 'Test Accuracy': test_acc}

# Models that benefit from scaling (use pipeline)
knn = Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier())])
svm = Pipeline([('scaler', StandardScaler()), ('clf', SVC())])
# LR and DT as before (LR often scaled too for fairness; DT doesn't need it)
lr_pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
dt = DecisionTreeClassifier(random_state=42)

results = [
    fit_and_score('Logistic Regression', lr_pipe, X_train, y_train, X_test, y_test),
    fit_and_score('KNN', knn, X_train, y_train, X_test, y_test),
    fit_and_score('Decision Tree', dt, X_train, y_train, X_test, y_test),
    fit_and_score('SVM', svm, X_train, y_train, X_test, y_test),
]
comparison_df = pd.DataFrame(results)
comparison_df

In [ ]:
# Format for display
comparison_df.round(4)

### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score

# Use F1 (or scoring='f1') to balance precision/recall when classes are imbalanced
f1_scorer = make_scorer(f1_score, pos_label=1)

In [ ]:
# Grid search for KNN
param_grid_knn = {'clf__n_neighbors': [5, 10, 20, 50], 'clf__weights': ['uniform', 'distance']}
grid_knn = GridSearchCV(knn, param_grid_knn, cv=5, scoring=f1_scorer, n_jobs=-1)
grid_knn.fit(X_train, y_train)
print("Best KNN params:", grid_knn.best_params_)
print("Best CV F1:", round(grid_knn.best_score_, 4))
print("Test F1:", round(f1_score(y_test, grid_knn.predict(X_test), pos_label=1), 4))

In [ ]:
# Grid search for Decision Tree
param_grid_dt = {'max_depth': [3, 5, 10, 20, None], 'min_samples_leaf': [1, 5, 10]}
grid_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid_dt, cv=5, scoring=f1_scorer, n_jobs=-1)
grid_dt.fit(X_train, y_train)
print("Best Decision Tree params:", grid_dt.best_params_)
print("Best CV F1:", round(grid_dt.best_score_, 4))
print("Test F1:", round(f1_score(y_test, grid_dt.predict(X_test), pos_label=1), 4))

In [ ]:
# Grid search for SVM (smaller grid - SVM can be slow)
param_grid_svm = {'clf__C': [0.1, 1, 10], 'clf__kernel': ['rbf', 'linear']}
grid_svm = GridSearchCV(svm, param_grid_svm, cv=3, scoring=f1_scorer, n_jobs=-1)
grid_svm.fit(X_train, y_train)
print("Best SVM params:", grid_svm.best_params_)
print("Best CV F1:", round(grid_svm.best_score_, 4))
print("Test F1:", round(f1_score(y_test, grid_svm.predict(X_test), pos_label=1), 4))

In [ ]:
# Compare tuned models (accuracy and F1)
tuned_results = []
for name, model in [('KNN (tuned)', grid_knn), ('Decision Tree (tuned)', grid_dt), ('SVM (tuned)', grid_svm)]:
    tuned_results.append({
        'Model': name,
        'Test Accuracy': accuracy_score(y_test, model.predict(X_test)),
        'Test F1': f1_score(y_test, model.predict(X_test), pos_label=1)
    })
pd.DataFrame(tuned_results).round(4)

In [ ]:
# Summary: Using F1 (or ROC-AUC) is often better than accuracy when the positive class (subscriptions) is rare.
# Hyperparameter tuning improved performance; best model can be chosen by cross-validated F1.

##### Questions